In [79]:
# https://judge.nitro-ai.org/competitions/nitro-2025/nitro-warmup-round/1/view

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [80]:
train = pd.read_csv("/kaggle/input/datasets/abukanabek/nitro-ai-warmup-1/train_data.csv")
test = pd.read_csv('/kaggle/input/datasets/abukanabek/nitro-ai-warmup-1/test_data.csv')
subm = pd.read_csv("/kaggle/input/datasets/abukanabek/nitro-ai-warmup-1/sample_output.csv")

test = test.sort_values('AppID').reset_index(drop=True)
subm = subm.sort_values(['subtaskID', 'datapointID']).reset_index(drop=True)

train.shape, test.shape, subm.shape

((3000, 11), (600, 10), (1200, 3))

In [81]:
train.head()

,AppID,Name,Release date,Estimated owners,Price,Metacritic score,Recommendations,Positive,Negative,Publishers,Genres
0,350110,TransOcean 2: Rivals,"May 10, 2016",100000 - 200000,29.99,69,428,305,281,astragon Entertainment,"Simulation,Strategy"
1,312780,Way of the Samurai 4,"July 23, 2015",200000 - 500000,24.99,72,1475,1288,318,"Spike Chunsoft Co., Ltd.","Action,Adventure"
2,230150,Incredipede,"March 18, 2013",50000 - 100000,9.99,74,140,243,96,Northway Games,"Action,Adventure,Casual,Indie,Simulation"
3,1252300,RetroMania Wrestling,"February 25, 2021",0 - 20000,29.99,66,174,211,30,Retrosoft Studios,"Action,Sports"
4,760650,Hammerting,"November 16, 2021",100000 - 200000,9.99,64,1358,1328,705,Team17 Digital,"Adventure,Indie,RPG,Simulation,Strategy"


In [82]:
def process_df(df):
    df['Release date'] = pd.to_datetime(df['Release date'])
    df['re-month'] = df['Release date'].dt.month
    df['re-day'] = df['Release date'].dt.day
    df['re-year'] = df['Release date'].dt.year
    df['est_min'] = df['Estimated owners'].map(lambda x: int(x.split('-')[0].strip()))
    df['est_max'] = df['Estimated owners'].map(lambda x: int(x.split('-')[1].strip()))
    df['est_dif'] = df['est_max'] - df['est_min']
    df['Genres'] = df['Genres'].map(lambda x: x.replace(',', ' '))

process_df(train)
process_df(test)

In [83]:
from sklearn.model_selection import train_test_split

features = [c for c in train.columns if c not in 
            ['AppID', 'Name', 'Release date', 'Estimated owners', 'Price', 'Publishers']]
text_features = ['Genres']
target_col = 'Price'

X, y = train[features], train[target_col]
X_test = test[features]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_valid.shape

((2400, 11), (600, 11))

In [84]:
from catboost import Pool

train_pool = Pool(X_train, y_train, text_features=text_features)
valid_pool = Pool(X_valid, y_valid, text_features=text_features)

In [85]:
from catboost import CatBoostRegressor

params = {
    'iterations': 2000,
    'loss_function': 'MAE',
    'eval_metric': 'MAE',
    'metric_period': 200,
    'max_depth': 4,
    'random_state': 42,
}

model = CatBoostRegressor(**params)

model.fit(train_pool, eval_set=valid_pool)

0:	learn: 7.9929449	test: 8.1142491	best: 8.1142491 (0)	total: 1.9ms	remaining: 3.8s
200:	learn: 5.9699997	test: 6.4384537	best: 6.4384537 (200)	total: 257ms	remaining: 2.3s
400:	learn: 5.5928170	test: 6.3576173	best: 6.3576173 (400)	total: 504ms	remaining: 2.01s
600:	learn: 5.3071364	test: 6.3093335	best: 6.3093335 (600)	total: 754ms	remaining: 1.75s
800:	learn: 5.1246070	test: 6.2857752	best: 6.2857752 (800)	total: 1s	remaining: 1.5s
1000:	learn: 4.9777717	test: 6.2779071	best: 6.2779071 (1000)	total: 1.26s	remaining: 1.26s
1200:	learn: 4.8617382	test: 6.2778355	best: 6.2778355 (1200)	total: 1.51s	remaining: 1.01s
1400:	learn: 4.7510351	test: 6.2738692	best: 6.2738692 (1400)	total: 1.79s	remaining: 765ms
1600:	learn: 4.6685517	test: 6.2764358	best: 6.2738692 (1400)	total: 2.07s	remaining: 515ms
1800:	learn: 4.5926873	test: 6.2789074	best: 6.2738692 (1400)	total: 2.32s	remaining: 257ms
1999:	learn: 4.5351626	test: 6.2697034	best: 6.2697034 (1999)	total: 2.57s	remaining: 0us

bestTest 

In [86]:
y_pred = model.predict(X_test)

subm['answer'] = subm['answer'].astype('float64')
subm.iloc[:len(test), 2] = (test['est_min'] + test['est_max']) / 2
subm.iloc[len(test):, 2] = y_pred

subm.to_csv("submission.csv", index=False)

subm.head()

,subtaskID,datapointID,answer
0,1,100,15000000.0
1,1,130,15000000.0
2,1,240,15000000.0
3,1,550,35000000.0
4,1,620,15000000.0
